This notebook:
- Defines which customers are churned (no purchase in 90 days)
- Engineers features from customer behaviour
- Trains Logistic Regression and Random Forest models
- Prints precision / recall / F1 report
- Draws **7 interactive Plotly charts** — confusion matrix, ROC, feature importance, and more

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [8]:
PLOTLY_LAYOUT = dict(
    template      = 'plotly_dark',
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#1a1a2e',
    font          = dict(family='monospace', color='#ccc'),
    margin        = dict(t=60, b=40, l=40, r=40),
)

PURPLE = '#7c6fcd'
GREEN  = '#4ecb8d'
ORANGE = '#f4845f'
RED    = '#e05c6b'

**Churn definition:** a customer is churned if they have NOT purchased in the last **90 days**.
This is a standard retail threshold — most loyal customers buy at least once per quarter.

In [9]:
# Load and Label data
df = pd.read_csv("customer_summary.csv")
print(f"{len(df):,} customers loaded")
df.head()

4,338 customers loaded


,CustomerID,first_purchase,last_purchase,total_orders,total_items,total_revenue,avg_order_value,unique_products,unique_countries,recency_days,tenure_days
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,1,74215,77183.60,77183.600000,1,1,326,326
1,12347,2010-12-07 14:57:00,2011-12-07 15:52:00,7,2458,4310.00,23.681319,103,1,2,367
2,12348,2010-12-16 19:09:00,2011-09-25 13:13:00,4,2341,1797.24,57.975484,22,1,75,358
3,12349,2011-11-21 09:51:00,2011-11-21 09:51:00,1,631,1757.55,24.076027,73,1,19,19
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,19.670588,17,1,310,310


In [10]:
# Define Churn

CHURN_DAYS = 90
df["churned"] = (df["recency_days"] > CHURN_DAYS).astype(int)

churn_rate = df["churned"].mean() * 100
print(f"\nChurn Definition:    no purchase in last {CHURN_DAYS} days")
print(f"Churned customers:    {df['churned'].sum():,}  ({churn_rate:.1f}%)")
print(f"Retained customers:   {(df['churned'] == 0).sum():,}  ({100-churn_rate:.1f}%)")


Churn Definition:    no purchase in last 90 days
Churned customers:    1,449  (33.4%)
Retained customers:   2,889  (66.6%)


**Churn Breakdown**

In [11]:
churn_counts = df['churned'].value_counts().reset_index()
churn_counts.columns = ['Churned', 'Count']
churn_counts['Label'] = churn_counts['Churned'].map({0: 'Retained', 1: 'Churned'})

fig = go.Figure(go.Pie(
    labels        = churn_counts['Label'],
    values        = churn_counts['Count'],
    hole          = 0.5,
    marker_colors = [GREEN, RED],
    textinfo      = 'percent+label',
    hovertemplate = '<b>%{label}</b><br>Customers: %{value:,}<br>Share: %{percent}<extra></extra>',
))

fig.add_annotation(
    text      = f'{churn_rate:.1f}%<br>churn rate',
    x=0.5, y=0.5,
    showarrow = False,
    font      = dict(size=15, color='#eee'),
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text=f'Churned vs retained customers (threshold = {CHURN_DAYS} days)', font=dict(size=16)),
    height = 400,
)

fig.show()

Features are the input columns the model learns from.
We use 7 direct columns plus 3 derived ratios we calculate.

In [12]:
# Feature Engineering
#   We create features the model will use to predict churn.
#   Every feature is based on observable customer behaviour.

# Derived ratio features
df['orders_per_month']  = df['total_orders']  / (df['tenure_days'] / 30).clip(lower=1)
df['revenue_per_order'] = df['total_revenue'] / df['total_orders'].clip(lower=1)
df['items_per_order']   = df['total_items']   / df['total_orders'].clip(lower=1)

FEATURES = [
    'recency_days',      # days since last purchase — strongest signal
    'total_orders',      # how often they buy
    'total_items',       # total units ever bought
    'total_revenue',     # total money spent
    'avg_order_value',   # average spend per order
    'unique_products',   # variety of products bought
    'tenure_days',       # how long they have been a customer
    'orders_per_month',  # purchase velocity (derived)
    'revenue_per_order', # spend per order (derived)
    'items_per_order',   # basket size (derived)
]

X = df[FEATURES].fillna(0)
y = df['churned']

print(f'Features: {len(FEATURES)}')
print(f'X shape:  {X.shape}  (rows = customers, cols = features)')
X.describe().round(2)

Features: 10
X shape:  (4338, 10)  (rows = customers, cols = features)


,recency_days,total_orders,total_items,total_revenue,avg_order_value,unique_products,tenure_days,orders_per_month,revenue_per_order,items_per_order
count,4338.00,4338.00,4338.00,4338.00,4338.00,4338.00,4338.00,4338.00,4338.00,4338.00
mean,92.54,4.27,1191.29,2054.27,68.35,61.50,223.31,0.64,419.17,253.48
std,100.01,7.70,5046.08,8989.23,1467.92,85.37,117.89,0.74,1796.54,1312.91
min,1.00,1.00,1.00,3.75,2.10,1.00,1.00,0.08,3.45,1.00
25%,18.00,1.00,160.00,307.41,12.37,16.00,113.00,0.23,178.62,93.00
50%,51.00,2.00,379.00,674.48,17.72,35.00,249.00,0.45,293.90,161.75
75%,142.00,5.00,992.75,1661.74,24.86,77.00,327.00,0.81,430.11,272.00
max,374.00,209.00,196915.00,280206.02,77183.60,1787.00,374.00,16.76,84236.25,74215.00


In [13]:
# Train/Test Split
# **80% training** — the model learns patterns from this
# **20% test** — hidden from the model during training, used only to measure real performance

#`stratify=y` keeps the churn ratio (33%) identical in both halves.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")


Train: 3,470  |  Test: 868


# Train Two models and compare

Both models are wrapped in a **Pipeline** that:
1. Scales all features to the same range (`StandardScaler`)
2. Fits the model on the scaled data

In [32]:
# MODEL A: Logistic Regression
# Simple linear model — fast, interpretable, good baseline

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),         # normalise features
    ("model",  LogisticRegression(max_iter=500, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
lr_pred  = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)
print(f"Logistic Regression  AUC = {lr_auc:.4f}")



Logistic Regression  AUC = 0.9999


In [33]:
# MODEL B: Random Forest

# Ensemble of 200 decision trees — handles non-linear patterns, usually better

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  RandomForestClassifier(
        n_estimators=200,       # 200 trees
        max_depth=8,            # don't overfit
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1               # use all CPU cores
    ))
])
rf_pipeline.fit(X_train, y_train)
rf_pred  = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)
print(f"Random Forest        AUC = {rf_auc:.4f}")



Random Forest        AUC = 1.0000


In [34]:
# Pick the best model
if rf_auc >= lr_auc:
    best_model, best_pred, best_proba = rf_pipeline, rf_pred, rf_proba
    best_name = "Random Forest"
else:
    best_model, best_pred, best_proba = lr_pipeline, lr_pred, lr_proba
    best_name = "Logistic Regression"
print(f"\nBest model: {best_name}  (AUC = {max(rf_auc, lr_auc):.4f})")


Best model: Random Forest  (AUC = 1.0000)


# Classification Report

| Metric | What it means |
|---|---|
| **Precision** | Of all customers predicted as churned, how many actually churned? |
| **Recall** | Of all customers who actually churned, how many did we catch? |
| **F1** | Balanced average of precision and recall |

In [35]:
# classification report

print("\nClassification Report — Random Forest:")
print(classification_report(y_test, rf_pred, target_names=["Retained","Churned"]))

print('=' * 55)

print("Classification Report — Logistic Regression:")
print(classification_report(y_test, lr_pred, target_names=["Retained","Churned"]))



Classification Report — Random Forest:
              precision    recall  f1-score   support

    Retained       1.00      1.00      1.00       578
     Churned       1.00      1.00      1.00       290

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

Classification Report — Logistic Regression:
              precision    recall  f1-score   support

    Retained       0.99      1.00      0.99       578
     Churned       1.00      0.97      0.99       290

    accuracy                           0.99       868
   macro avg       0.99      0.99      0.99       868
weighted avg       0.99      0.99      0.99       868



#Precision / Recall / F1 comparison (grouped bar)

Visual comparison of both models across all three metrics.

In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score

metrics_data = pd.DataFrame({
    'Metric':  ['Precision', 'Recall', 'F1 Score'] * 2,
    'Model':   ['Random Forest'] * 3 + ['Logistic Regression'] * 3,
    'Score':   [
        precision_score(y_test, rf_pred),
        recall_score(y_test, rf_pred),
        f1_score(y_test, rf_pred),
        precision_score(y_test, lr_pred),
        recall_score(y_test, lr_pred),
        f1_score(y_test, lr_pred),
    ]
})

fig = px.bar(
    metrics_data,
    x             = 'Metric',
    y             = 'Score',
    color         = 'Model',
    barmode       = 'group',
    color_discrete_map = {
        'Random Forest':      GREEN,
        'Logistic Regression': PURPLE,
    },
    text          = metrics_data['Score'].apply(lambda x: f'{x:.3f}'),
)

fig.update_traces(
    textposition  = 'outside',
    hovertemplate = '<b>%{x}</b> — %{fullData.name}<br>Score: %{y:.4f}<extra></extra>',
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Model comparison — Precision, Recall, F1', font=dict(size=16)),
    xaxis  = dict(title=''),
    yaxis  = dict(title='Score', range=[0, 1.12], gridcolor='#2a2a38'),
    height = 420,
)

fig.show()

# Confusion matrix (interactive heatmap)

In [37]:
cm = confusion_matrix(y_test, rf_pred)
tn, fp, fn, tp = cm.ravel()

labels = [['True Negative', 'False Positive'],
          ['False Negative', 'True Positive']]

meanings = [
    ['Correctly said Retained', 'Wrongly said Churned (false alarm)'],
    ['Missed a real churner (costly!)', 'Correctly said Churned']
]

fig = go.Figure(go.Heatmap(
    z             = cm,
    x             = ['Predicted Retained', 'Predicted Churned'],
    y             = ['Actual Retained', 'Actual Churned'],
    colorscale    = 'RdYlGn',
    showscale     = False,
    hovertemplate = (
        '<b>%{customdata[0]}</b><br>'
        '%{customdata[1]}<br>'
        'Count: %{z:,}<extra></extra>'
    ),
    customdata    = np.array([
        [(labels[i][j], meanings[i][j]) for j in range(2)]
        for i in range(2)
    ]),
))

# Annotate cells with count and label
for i in range(2):
    for j in range(2):
        fig.add_annotation(
            x         = j, y = i,
            text      = f'<b>{cm[i,j]:,}</b><br><span style="font-size:10px">{labels[i][j]}</span>',
            showarrow = False,
            font      = dict(size=13, color='black'),
        )

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Confusion matrix — Random Forest', font=dict(size=16)),
    xaxis  = dict(title='Predicted label', side='bottom'),
    yaxis  = dict(title='Actual label', autorange='reversed'),
    height = 380,
    width  = 560,
)

fig.show()

print(f'True Negatives  (correctly said Retained): {tn:,}')
print(f'False Positives (wrongly said Churned):    {fp:,}  ← wasted retention offers')
print(f'False Negatives (missed actual Churners):  {fn:,}  ← most costly mistake')
print(f'True Positives  (correctly said Churned):  {tp:,}')

True Negatives  (correctly said Retained): 578
False Positives (wrongly said Churned):    0  ← wasted retention offers
False Negatives (missed actual Churners):  1  ← most costly mistake
True Positives  (correctly said Churned):  289


#ROC curve (both models)
The ROC curve shows how well each model separates churned from retained customers.
AUC closer to 1.0 = better. 0.5 = random guessing.

In [38]:
fig = go.Figure()

for name, proba, color in [
    ('Random Forest',      rf_proba, GREEN),
    ('Logistic Regression', lr_proba, PURPLE),
]:
    fpr, tpr, thresholds = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    fig.add_trace(go.Scatter(
        x             = fpr,
        y             = tpr,
        mode          = 'lines',
        name          = f'{name}  (AUC = {auc:.3f})',
        line          = dict(color=color, width=2.5),
        hovertemplate = (
            f'<b>{name}</b><br>'
            'False Positive Rate: %{x:.3f}<br>'
            'True Positive Rate:  %{y:.3f}<extra></extra>'
        ),
    ))

# Diagonal baseline
fig.add_trace(go.Scatter(
    x          = [0, 1], y = [0, 1],
    mode       = 'lines',
    name       = 'Random guess (AUC = 0.5)',
    line       = dict(color='#555', dash='dash', width=1.5),
    hoverinfo  = 'skip',
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='ROC curve — both models vs random baseline', font=dict(size=16)),
    xaxis  = dict(title='False Positive Rate', gridcolor='#2a2a38', range=[0,1]),
    yaxis  = dict(title='True Positive Rate',  gridcolor='#2a2a38', range=[0,1]),
    height = 480,
    legend = dict(x=0.55, y=0.1),
)

fig.show()

# Precision-Recall curve
More informative than ROC when data is imbalanced.

Top-right corner = perfect model.

In [39]:
fig = go.Figure()

for name, proba, color in [
    ('Random Forest',       rf_proba, GREEN),
    ('Logistic Regression', lr_proba, PURPLE),
]:
    prec, rec, thresholds = precision_recall_curve(y_test, proba)
    fig.add_trace(go.Scatter(
        x             = rec,
        y             = prec,
        mode          = 'lines',
        name          = name,
        line          = dict(color=color, width=2.5),
        hovertemplate = (
            f'<b>{name}</b><br>'
            'Recall:    %{x:.3f}<br>'
            'Precision: %{y:.3f}<extra></extra>'
        ),
    ))

# Baseline = churn rate (random classifier)
baseline = y_test.mean()
fig.add_hline(
    y                  = baseline,
    line_dash          = 'dash',
    line_color         = '#555',
    annotation_text    = f'Random baseline ({baseline:.2f})',
    annotation_position= 'bottom right',
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Precision-Recall curve', font=dict(size=16)),
    xaxis  = dict(title='Recall',    gridcolor='#2a2a38', range=[0,1]),
    yaxis  = dict(title='Precision', gridcolor='#2a2a38', range=[0,1.05]),
    height = 460,
)

fig.show()
print('Top-right = perfect. The further from the dashed baseline, the better.')

Top-right = perfect. The further from the dashed baseline, the better.


In [44]:
importances = pd.Series(
    rf_pipeline.named_steps['model'].feature_importances_,
    index=FEATURES
).sort_values(ascending=True).reset_index()
importances.columns = ['Feature', 'Importance']

# Colour top feature differently
max_imp = importances['Importance'].max()
importances['Color'] = importances['Importance'].apply(
    lambda x: ORANGE if x == max_imp else PURPLE
)

fig = go.Figure(go.Bar(
    x            = importances['Importance'],
    y            = importances['Feature'],
    orientation  = 'h',
    marker_color = importances['Color'],
    text         = importances['Importance'].apply(lambda x: f'{x:.4f}'),
    textposition = 'outside',
    hovertemplate = '<b>%{y}</b><br>Importance: %{x:.4f}<extra></extra>',
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title      = dict(text='Feature importance — Random Forest', font=dict(size=16)),
    xaxis      = dict(title='Importance score', gridcolor='#2a2a38'),
    yaxis      = dict(title=''),
    height     = 440,
    showlegend = False,
)

fig.show()
top_feature = importances.loc[importances['Importance'].idxmax(), 'Feature']
print(f'💡 Most important feature: {top_feature}')
print('   This makes sense — recency directly determines the churn label (>90 days).')

💡 Most important feature: recency_days
   This makes sense — recency directly determines the churn label (>90 days).


Score all customers and save predictions

In [47]:
df['churn_probability'] = rf_pipeline.predict_proba(X)[:, 1]
df['churn_predicted']   = rf_pipeline.predict(X)

def risk_tier(p):
    if p >= 0.75: return 'High Risk'
    if p >= 0.45: return 'Medium Risk'
    return 'Low Risk'

df['risk_tier'] = df['churn_probability'].apply(risk_tier)

# Removed PROJECT_FOLDER and directly specified the filename
output_cols = ['CustomerID','recency_days','total_orders','total_revenue',
               'churn_probability','churn_predicted','risk_tier']

df[output_cols].sort_values('churn_probability', ascending=False).to_csv('churn_predictions.csv', index=False)

print(f'Saved: churn_predictions.csv')
print()
print('Risk tier breakdown:')
print(df['risk_tier'].value_counts().to_string())

Saved: churn_predictions.csv

Risk tier breakdown:
risk_tier
Low Risk       2889
High Risk      1447
Medium Risk       2


# Churn probability distribution
Shows how the model scores retained vs churned customers.
Good separation = the two peaks should be on opposite sides of 0.5.

In [49]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x             = df[df['churned'] == 0]['churn_probability'],
    name          = 'Retained',
    marker_color  = GREEN,
    opacity       = 0.75,
    nbinsx        = 40,
    hovertemplate = 'Probability: %{x:.2f}<br>Retained customers: %{y:,}<extra></extra>',
))

fig.add_trace(go.Histogram(
    x             = df[df['churned'] == 1]['churn_probability'],
    name          = 'Churned',
    marker_color  = RED,
    opacity       = 0.75,
    nbinsx        = 40,
    hovertemplate = 'Probability: %{x:.2f}<br>Churned customers: %{y:,}<extra></extra>',
))

fig.add_vline(
    x                  = 0.5,
    line_dash          = 'dash',
    line_color         = ORANGE,
    line_width         = 2,
    annotation_text    = 'Decision threshold (0.5)',
    annotation_position= 'top right',
    annotation_font    = dict(color=ORANGE),
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title    = dict(text='Churn probability distribution — retained vs churned', font=dict(size=16)),
    xaxis    = dict(title='Predicted churn probability', gridcolor='#2a2a38'),
    yaxis    = dict(title='Number of customers',         gridcolor='#2a2a38'),
    barmode  = 'overlay',
    height   = 430,
)

fig.show()
print('   Green should cluster near 0 (low churn risk).')
print('   Red should cluster near 1 (high churn risk).')
print('   Clean separation = a well-calibrated model.')

   Green should cluster near 0 (low churn risk).
   Red should cluster near 1 (high churn risk).
   Clean separation = a well-calibrated model.


#Top at-risk customers preview

In [50]:
top_risk = (
    df[df['risk_tier'] == 'High Risk']
    .sort_values('churn_probability', ascending=False)
    .head(15)[['CustomerID','recency_days','total_orders',
               'total_revenue','churn_probability','risk_tier']]
    .copy()
)

top_risk['churn_probability'] = top_risk['churn_probability'].round(3)
top_risk['total_revenue']     = top_risk['total_revenue'].round(2)

print('Top 15 highest-risk customers:')
top_risk

Top 15 highest-risk customers:


,CustomerID,recency_days,total_orders,total_revenue,churn_probability,risk_tier
23,12373,311,1,364.60,1.0,High Risk
4334,18281,181,1,80.82,1.0,High Risk
2639,15942,134,1,337.44,1.0,High Risk
2640,15944,277,1,325.10,1.0,High Risk
2641,15945,365,1,181.00,1.0,High Risk
2656,15970,304,1,314.10,1.0,High Risk
2658,15973,367,1,307.82,1.0,High Risk
2681,16006,211,1,101.40,1.0,High Risk
2703,16030,310,1,331.24,1.0,High Risk
2609,15895,150,1,179.17,1.0,High Risk
